In [ ]:
import tensorflow as tf

np.random.seed(42)
TF_SEED = 42
tf.random.set_seed(TF_SEED)

T = 6000
t = np.arange(T)
series = np.sin(0.02 * t) + 0.3*np.sin(0.15 * t) + 0.1*np.random.randn(T)

# Hyperparams
SEQ_LEN = 30          # number of past steps for forecasting
FULL_LEN = SEQ_LEN+1  # window + next target value
Z_DIM = 32            # generator noise vector size
BATCH_SIZE = 64
GAN_EPOCHS = 1500     # keep small for demo; increase for better samples
N_SYNTH = 3000        # how many synthetic sequences to add

# --------------------------------
# 1) Build sliding windows
# --------------------------------
def make_sequences(series, full_len):
    X = []
    for i in range(len(series) - full_len + 1):
        X.append(series[i:i+full_len])
    X = np.array(X)  # shape: (n_samples, FULL_LEN)
    return X

all_full = make_sequences(series, FULL_LEN)  # (n, 31)

# Train/test split (chronological)
split = int(0.8 * len(all_full))
train_full = all_full[:split]
test_full  = all_full[split:]

# Split into (window, target) for LSTM forecaster later
def split_xy(full):
    X = full[:, :SEQ_LEN]       # (n, 30)
    y = full[:, -1]             # (n,)
    return X[..., None], y      # add feature dim -> (n, 30, 1), (n,)

X_train_real, y_train_real = split_xy(train_full)
X_test, y_test = split_xy(test_full)

# --------------------------------
# 2) GAN components
#    - Generator: noise -> (FULL_LEN, 1)
#    - Discriminator: (FULL_LEN, 1) -> real/fake
# --------------------------------
def build_generator(z_dim, full_len):
    z = layers.Input(shape=(z_dim,))
    x = layers.Dense(128, activation='relu')(z)
    x = layers.Dense(full_len*64, activation='relu')(x)
    x = layers.Reshape((full_len, 64))(x)
    x = layers.LSTM(64, return_sequences=True)(x)
    out = layers.TimeDistributed(layers.Dense(1))(x)
    return Model(z, out, name="Generator")

def build_discriminator(full_len):
    seq_in = layers.Input(shape=(full_len, 1))
    x = layers.LSTM(64)(seq_in)
    x = layers.Dense(32, activation='relu')(x)
    out = layers.Dense(1, activation='sigmoid')(x)
    return Model(seq_in, out, name="Discriminator")

G = build_generator(Z_DIM, FULL_LEN)
D = build_discriminator(FULL_LEN)

D.compile(
    optimizer=tf.keras.optimizers.Adam(1e-4),
    loss='binary_crossentropy',
    metrics=['accuracy']
)

# Combined model: G(z) -> D(G(z))
z_in = layers.Input(shape=(Z_DIM,))
fake_seq = G(z_in)
D.trainable = False
validity = D(fake_seq)
GAN = Model(z_in, validity, name="GAN")
GAN.compile(optimizer=tf.keras.optimizers.Adam(1e-4), loss='binary_crossentropy')

# --------------------------------
# 3) Prep real data for GAN (use full sequences)
# --------------------------------
train_full_seq = train_full[..., None]  # (n, FULL_LEN, 1)

# --------------------------------
# 4) Train GAN
# --------------------------------
real_label = np.ones((BATCH_SIZE, 1))
fake_label = np.zeros((BATCH_SIZE, 1))

n_batches = len(train_full_seq) // BATCH_SIZE

for epoch in range(1, GAN_EPOCHS+1):
    # --- Train Discriminator ---
    # sample real
    idx = np.random.randint(0, len(train_full_seq), BATCH_SIZE)
    real_seqs = train_full_seq[idx]

    # sample fake
    z = np.random.randn(BATCH_SIZE, Z_DIM)
    gen_seqs = G.predict(z, verbose=0)

    # train on real + fake
    d_loss_real = D.train_on_batch(real_seqs, real_label)
    d_loss_fake = D.train_on_batch(gen_seqs, fake_label)
    d_loss = 0.5 * (d_loss_real[0] + d_loss_fake[0])

    # --- Train Generator (wants discriminator to output 'real' on fakes) ---
    z = np.random.randn(BATCH_SIZE, Z_DIM)
    g_loss = GAN.train_on_batch(z, real_label)

    if epoch % 200 == 0:
        print(f"Epoch {epoch:4d} | D loss: {d_loss:.4f} | G loss: {g_loss:.4f}")

# --------------------------------
# 5) Sample synthetic sequences and split to (X, y)
# -------------------------------
# Optional: match scale of real data more tightly (simple clipping)
lo, hi = np.percentile(train_full, [1, 99])
X_synth = np.clip(X_synth, lo, hi)
y_synth = np.clip(y_synth, lo, hi)

# --------------------------------
# 6) Train LSTM forecaster on augmented data
# --------------------------------
X_train_aug = np.concatenate([X_train_real, X_synth], axis=0)
y_train_aug = np.concatenate([y_train_real, y_synth], axis=0)

NameError: name 'tf' is not defined

In [ ]:
# https://github.com/sdv-dev/TGAN/blob/master/examples/Usage_Example.ipynb